# Gaza Damage Assessment -- prototype pipeline (Appendix A development history)

<a href="https://colab.research.google.com/github/REPLACE_ME/Gaza-Damage-Project/blob/paper-aligned-pipeline/notebooks/01_prototype_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook is the cleaned-up, English-language version of this repo's
original prototype work (previously four overlapping, partly-Arabic-commented
notebooks). It is a **two-stage pipeline**:

1. **Building detection**: OpenStreetMap building footprints -> ResNet34-encoder
   U-Net, trained on Al-Shati Camp imagery.
2. **Damage detection**: a naive pixel-difference pseudo-label (before/after
   RGB change, thresholded, water-masked) -> EfficientNet-B4-encoder
   six-channel U-Net.
3. A final cell combines both models' predictions into a single overlay
   (yellow building outlines, red damage regions) and estimates the percentage
   of damage falling inside detected buildings.

**This is the prototype the paper's Appendix A describes as superseded**, not
the paper's final methodology -- it depends on a separately-trained building
model (error propagation risk), uses a heuristic pixel-difference pseudo-label
rather than UNOSAT weak labels, and doesn't do spatially isolated evaluation
or threshold calibration. See `notebooks/02_geodamage_ai_pipeline.ipynb` for
the paper's actual final design. This notebook is kept because it documents
real, working development history, not because it's the recommended pipeline.

Runs on Google Colab with a GPU (same pattern as the original notebooks:
install deps in the first cell, mount Drive for persistence between cells).


In [ ]:
# ═══════════════════════════════════════════════
# CELL 0 -- Install dependencies + download data
# ═══════════════════════════════════════════════

# Install required libraries
!pip install segmentation-models-pytorch albumentations kaggle osmnx rasterio geopandas -q

# Download the dataset from Kaggle
import os
from google.colab import files
# Upload your kaggle.json API credentials file when prompted
uploaded = files.upload()
os.makedirs('/root/.config/kaggle', exist_ok=True)
os.rename('kaggle.json', '/root/.config/kaggle/kaggle.json')
os.chmod('/root/.config/kaggle/kaggle.json', 0o600)

# Download the dataset
!kaggle datasets download -d abdoomoh/gaza-before-and-after-2 -p /content/data --unzip

# (Alternative, no Kaggle credentials needed: scripts/download_imagery.py in this
# repo downloads a targeted subset of sections anonymously -- see data/README.md.)


In [ ]:
# ═══════════════════════════════════════════════
# CELL 1 -- Explore the data + build Al-Shati Camp before/after pairs
# ═══════════════════════════════════════════════

import json
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict
import pickle
from google.colab import drive
drive.mount('/content/drive')

DATA_PATH = Path('/content/data/Gaza Before and After')
SAVE_PATH = '/content/drive/MyDrive/Shati_Project'
os.makedirs(SAVE_PATH, exist_ok=True)
CUTOFF = '2023-10-07'

# Approximate Al-Shati Camp bounding box
SHATI = {'west': 34.425, 'south': 31.515, 'east': 34.465, 'north': 31.545}

# ─── Helper functions ─────────────────────────────
def is_valid(path):
    img = cv2.imread(str(path))
    if img is None: return False
    if img.mean() < 10: return False  # near-black -> no-data/placeholder tile
    return True

def in_shati(bbox):
    try:
        return (bbox[0] >= SHATI['west']  - 0.05 and
                bbox[2] <= SHATI['east']  + 0.05 and
                bbox[1] >= SHATI['south'] - 0.05 and
                bbox[3] <= SHATI['north'] + 0.05)
    except:
        return False

# ─── 1. Collect metadata ──────────────────────────
print("Reading metadata...")
all_meta = []
for json_file in DATA_PATH.glob('**/*.json'):
    try:
        with open(json_file) as f:
            meta = json.load(f)
        section  = meta['section_id']
        date     = meta['timestamp']
        img_path = DATA_PATH / 'images' / section / f'{date}.png'
        meta['img_path']   = str(img_path)
        # default bbox if missing from this entry's metadata
        if 'bbox' not in meta:
            meta['bbox'] = [0, 0, 0, 0]
        meta['img_exists'] = img_path.exists() and is_valid(img_path)
        all_meta.append(meta)
    except:
        continue

all_meta.sort(key=lambda x: x['timestamp'])
total  = len(all_meta)
valid  = sum(1 for m in all_meta if m['img_exists'])
print(f"Total: {total} | valid: {valid}")

# ─── 2. Filter to Al-Shati Camp ───────────────────
shati_meta = [
    m for m in all_meta
    if m['img_exists'] and m.get('bbox') and in_shati(m['bbox'])
]
print(f"Al-Shati Camp images: {len(shati_meta)}")

# If too few -- widen the search radius
if len(shati_meta) < 5:
    print("Too few -- widening the search area...")
    shati_meta = [
        m for m in all_meta
        if m['img_exists']
        and m['bbox'] != [0,0,0,0]
        and abs((m['bbox'][0]+m['bbox'][2])/2 - 34.44) < 0.15
        and abs((m['bbox'][1]+m['bbox'][3])/2 - 31.53) < 0.12
    ]
    print(f"After widening: {len(shati_meta)}")

# If still nothing -- fall back to using all available data
if len(shati_meta) == 0:
    print("No Al-Shati Camp images found -- using all available data")
    shati_meta = [m for m in all_meta if m['img_exists']]

# ─── 3. Split pre / post ──────────────────────────
pre_meta  = [m for m in shati_meta if m['timestamp'] <  CUTOFF]
post_meta = [m for m in shati_meta if m['timestamp'] >= CUTOFF]

print(f"\npre  (before the war): {len(pre_meta)}")
print(f"post (after the war):  {len(post_meta)}")

# ─── 4. Build the best pre/post pair per section ──
pre_by_section  = defaultdict(list)
post_by_section = defaultdict(list)
for m in pre_meta:  pre_by_section[m['section_id']].append(m)
for m in post_meta: post_by_section[m['section_id']].append(m)

pairs = []
for section_id in pre_by_section:
    if section_id not in post_by_section:
        continue
    best_pre  = max(pre_by_section[section_id],  key=lambda x: x['timestamp'])  # latest pre-war date
    best_post = min(post_by_section[section_id], key=lambda x: x['timestamp'])  # earliest post-war date
    pairs.append({
        'section': section_id,
        'pre':     best_pre,
        'post':    best_post
    })

print(f"\nPairs ready: {len(pairs)}")

# ─── 5. Visualize the pairs ───────────────────────
if pairs:
    show = min(len(pairs), 4)
    fig, axes = plt.subplots(show, 2, figsize=(10, 5*show))
    if show == 1: axes = [axes]
    for i in range(show):
        pair     = pairs[i]
        pre_img  = cv2.cvtColor(cv2.imread(pair['pre']['img_path']),  cv2.COLOR_BGR2RGB)
        post_img = cv2.cvtColor(cv2.imread(pair['post']['img_path']), cv2.COLOR_BGR2RGB)
        axes[i][0].imshow(pre_img)
        axes[i][0].set_title(f"Pre — {pair['pre']['timestamp']}\n{pair['section']}", fontsize=10)
        axes[i][0].axis('off')
        axes[i][1].imshow(post_img)
        axes[i][1].set_title(f"Post — {pair['post']['timestamp']}", fontsize=10)
        axes[i][1].axis('off')
    plt.suptitle("Al-Shati Camp — Pre/Post pairs", fontsize=13)
    plt.tight_layout()
    plt.savefig("shati_pairs.png", dpi=150, bbox_inches='tight')
    plt.show()

# ─── 6. Save ───────────────────────────────────────
with open(f'{SAVE_PATH}/shati_pairs.pkl', 'wb') as f:
    pickle.dump(pairs, f)

with open(f'{SAVE_PATH}/all_meta.pkl', 'wb') as f:
    pickle.dump(all_meta, f)

print(f"\nSaved: {len(pairs)} pairs to Drive!")
if pairs:
    print(f"Oldest pre:  {min(p['pre']['timestamp']  for p in pairs)}")
    print(f"Newest post: {max(p['post']['timestamp'] for p in pairs)}")


In [ ]:
# ═══════════════════════════════════════════════
# CELL 2 -- Prepare building-detection training data
# ═══════════════════════════════════════════════

import cv2
import numpy as np
import json
import os
import pickle
import osmnx as ox
import matplotlib.pyplot as plt
from pathlib import Path

SAVE_PATH = '/content/drive/MyDrive/Shati_Project'
DATA_PATH = Path('/content/data/Gaza Before and After')

# ─── Load the before/after pairs ──────────────────
with open(f'{SAVE_PATH}/shati_pairs.pkl', 'rb') as f:
    pairs = pickle.load(f)
print(f"{len(pairs)} pairs loaded!")

# ─── 1. Load Al-Shati Camp buildings from OSM ─────
print("Downloading buildings from OpenStreetMap...")
buildings = ox.features_from_bbox(
    bbox=(34.415, 31.510, 34.475, 31.550),
    tags={'building': True}
)
buildings = buildings[
    buildings.geometry.geom_type.isin(['Polygon','MultiPolygon'])
].copy().to_crs('EPSG:4326').reset_index(drop=True)
print(f"{len(buildings)} buildings from OSM!")

# ─── 2. Helper functions ──────────────────────────
def geo_to_pixel(lon, lat, bbox, w, h):
    x = int((lon - bbox[0]) / (bbox[2] - bbox[0]) * w)
    y = int((bbox[3] - lat) / (bbox[3] - bbox[1]) * h)
    return max(0,min(x,w-1)), max(0,min(y,h-1))

def make_building_mask(buildings, bbox, w, h):
    mask = np.zeros((h, w), dtype=np.uint8)
    for _, row in buildings.iterrows():
        try:
            geom  = row.geometry
            polys = list(geom.geoms) if geom.geom_type=='MultiPolygon' else [geom]
            for poly in polys:
                pts = np.array([
                    geo_to_pixel(lon, lat, bbox, w, h)
                    for lon, lat in poly.exterior.coords
                ], dtype=np.int32)
                cv2.fillPoly(mask, [pts], 1)
        except:
            continue
    return mask

# ─── 3. Build the training patches ────────────────
TARGET_SIZE = 256
STRIDE      = 64    # large overlap, to get more training patches
MAX_PATCHES = 3000

all_images = []
all_masks  = []

# Use every pre-war Al-Shati image, not just the one picked pair per section
from collections import defaultdict
with open(f'{SAVE_PATH}/all_meta.pkl', 'rb') as f:
    all_meta = pickle.load(f)

CUTOFF = '2023-10-07'
SHATI  = {'west':34.415, 'south':31.510, 'east':34.475, 'north':31.550}

def in_shati(bbox):
    try:
        cx = (bbox[0]+bbox[2])/2
        cy = (bbox[1]+bbox[3])/2
        return (SHATI['west']  <= cx <= SHATI['east'] and
                SHATI['south'] <= cy <= SHATI['north'])
    except:
        return False

# all pre-war Al-Shati images
shati_pre = [
    m for m in all_meta
    if m['img_exists']
    and m['timestamp'] < CUTOFF
    and m.get('bbox') and m['bbox'] != [0,0,0,0]
    and in_shati(m['bbox'])
]
print(f"Pre-war Al-Shati images: {len(shati_pre)}")

print("Building patches...")

for meta in shati_pre:
    if len(all_images) >= MAX_PATCHES:
        break

    img  = cv2.cvtColor(cv2.imread(meta['img_path']), cv2.COLOR_BGR2RGB)
    bbox = meta['bbox']
    h, w = img.shape[:2]

    # build the building mask for this image
    bldg_mask = make_building_mask(buildings, bbox, w, h)

    # skip tiles with (almost) no buildings
    if bldg_mask.sum() < 500:
        continue

    # cut into patches
    for y in range(0, h - TARGET_SIZE + 1, STRIDE):
        for x in range(0, w - TARGET_SIZE + 1, STRIDE):
            img_p  = img      [y:y+TARGET_SIZE, x:x+TARGET_SIZE]
            mask_p = bldg_mask[y:y+TARGET_SIZE, x:x+TARGET_SIZE]
            # only keep patches with a meaningful amount of building pixels
            if mask_p.sum() > 200:
                all_images.append(img_p.astype(np.float32)/255.0)
                all_masks.append(mask_p.astype(np.float32))

    if len(all_images) % 300 == 0 and len(all_images) > 0:
        print(f"  patches: {len(all_images)}")

print(f"\nTotal patches: {len(all_images)}")
print(f"Building-pixel fraction: {np.mean([m.mean() for m in all_masks])*100:.1f}%")

# ─── 4. Save to Drive ─────────────────────────────
np.save(f'{SAVE_PATH}/building_images.npy', np.array(all_images, dtype=np.float32))
np.save(f'{SAVE_PATH}/building_masks.npy',  np.array(all_masks,  dtype=np.float32))

# ─── 5. Example visualization ─────────────────────
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for i in range(2):
    idx = i * 50 + 10
    overlay = all_images[idx].copy()
    mask_3ch = np.stack([all_masks[idx]]*3, axis=-1)
    yellow = np.array([1.0, 0.84, 0.0], dtype=np.float32)
    for c in range(3):
        overlay[:,:,c][all_masks[idx] > 0] = (
            overlay[:,:,c][all_masks[idx] > 0] * 0.4 + yellow[c] * 0.6
        )

    axes[i][0].imshow(all_images[idx])
    axes[i][0].set_title("The original image"); axes[i][0].axis('off')

    axes[i][1].imshow(all_masks[idx], cmap='gray')
    axes[i][1].set_title("Mask Buildings (OSM)"); axes[i][1].axis('off')

    axes[i][2].imshow(overlay)
    axes[i][2].set_title("The buildings are in yellow"); axes[i][2].axis('off')

plt.suptitle(f"Building Detection Data — {len(all_images)} patch", fontsize=13)
plt.tight_layout()
plt.savefig("building_data.png", dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ═══════════════════════════════════════════════
# CELL 3 -- Train the building-detection model
# ═══════════════════════════════════════════════

import torch
import torch.nn as nn
import numpy as np
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import os
from sklearn.metrics import f1_score

SAVE_PATH = '/content/drive/MyDrive/Shati_Project'
device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device used for the building model: {device}")

# ─── 1. Load the data ─────────────────────────────
images = np.load(f'{SAVE_PATH}/building_images.npy')
masks  = np.load(f'{SAVE_PATH}/building_masks.npy')
print(f"{len(images)} patches ready for the building model!")
print(f"Patch shape: {images[0].shape}")

# ─── 2. Custom Dataset class ──────────────────────
class BuildingDataset(Dataset):
    def __init__(self, images, masks, augment=True):
        self.images = images
        self.masks  = masks
        # Deliberately heavy augmentation to improve accuracy and limit overfitting
        self.transform = A.Compose([
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.RandomRotate90(p=0.5),
            A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=15, p=0.5, border_mode=0),
            A.RandomBrightnessContrast(p=0.4),
            A.GaussNoise(p=0.2),
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2(),
        ]) if augment else A.Compose([
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2(),
        ])

    def __len__(self): return len(self.images)

    def __getitem__(self, i):
        img  = (self.images[i] * 255).astype(np.uint8)
        mask = self.masks[i].astype(np.float32)
        out  = self.transform(image=img, mask=mask)
        return out['image'], out['mask'].unsqueeze(0)

# ─── 3. Train/validation split (80/20) ────────────
split    = int(len(images) * 0.8)
train_ds = BuildingDataset(images[:split], masks[:split], augment=True)
val_ds   = BuildingDataset(images[split:], masks[split:], augment=False)

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=8, shuffle=False, num_workers=2)
print(f"Train: {len(train_ds)} patches | Val: {len(val_ds)} patches")

# ─── 4. Model architecture (ResNet34 encoder) ─────
model = smp.Unet(
    encoder_name    = "resnet34",
    encoder_weights = "imagenet",
    in_channels     = 3,
    classes         = 1,
).to(device)

# Loss: Dice for overall overlap + BCE for precise boundaries
dice_loss_fn = smp.losses.DiceLoss('binary')
bce_loss_fn  = nn.BCEWithLogitsLoss()

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
# Scheduler period matched to the full 50-epoch run for a stable end-of-training decay
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)

# ─── 5. Training loop (50 epochs) ─────────────────
best_loss  = float('inf')
train_hist = []
val_hist   = []

print("\nStarting training for 50 epochs...")

for epoch in range(1, 51):
    # training phase
    model.train()
    tr_loss = 0
    for imgs, msks in train_loader:
        imgs, msks = imgs.to(device), msks.to(device)
        optimizer.zero_grad()
        preds = model(imgs)
        loss  = dice_loss_fn(preds, msks) + 0.5 * bce_loss_fn(preds, msks)
        loss.backward()
        optimizer.step()
        tr_loss += loss.item()
    tr_loss /= len(train_loader)

    # validation phase (same combined loss as training, for a fair comparison)
    model.eval()
    vl_loss = 0
    with torch.no_grad():
        for imgs, msks in val_loader:
            imgs, msks = imgs.to(device), msks.to(device)
            preds_val = model(imgs)
            val_loss_combo = dice_loss_fn(preds_val, msks) + 0.5 * bce_loss_fn(preds_val, msks)
            vl_loss += val_loss_combo.item()
    vl_loss /= len(val_loader)

    scheduler.step()
    train_hist.append(tr_loss)
    val_hist.append(vl_loss)

    saved = ''
    if vl_loss < best_loss:
        best_loss = vl_loss
        torch.save(model.state_dict(), f'{SAVE_PATH}/best_building_model.pth')
        saved = '(best weights saved)'

    # print progress every 5 epochs, plus the first and last
    if epoch % 5 == 0 or epoch == 1 or epoch == 50:
        print(f"Epoch {epoch:02d}/50 | Train Loss: {tr_loss:.4f} | Val Loss: {vl_loss:.4f} {saved}")

print(f"\nBuilding model training complete! Best Val Loss: {best_loss:.4f}")

# ─── 6. Loss curve ─────────────────────────────────
plt.figure(figsize=(10,4))
plt.plot(train_hist, label='Train Loss', color='#8b5cf6', linewidth=2)
plt.plot(val_hist,   label='Val Loss',   color='#f59e0b', linewidth=2)
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.title('Training Curve — Building Detection Model (Shati Camp)', fontsize=12, fontweight='bold')
plt.legend(); plt.grid(alpha=0.3)
plt.savefig('building_curve.png', dpi=150, bbox_inches='tight')
plt.show()

# ─── 7. Evaluate the model + show final outputs and F1 Score ──
model.load_state_dict(torch.load(f'{SAVE_PATH}/best_building_model.pth'))
model.eval()

transform_test = A.Compose([
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

all_f1 = []
fig, axes = plt.subplots(3, 3, figsize=(14, 12))
# fixed random sample from the validation split, for reproducible test figures
np.random.seed(42)
test_idxs = np.random.choice(len(val_ds), 3, replace=False)

for i, idx in enumerate(test_idxs):
    real_idx = split + idx

    img_u8 = (images[real_idx] * 255).astype(np.uint8)
    tensor = transform_test(image=img_u8)['image'].unsqueeze(0).to(device)
    with torch.no_grad():
        pred = torch.sigmoid(model(tensor)).cpu().squeeze().numpy()
    pred_bin = (pred > 0.5).astype(np.float32)

    # pixel-wise F1-score against the OSM-derived building mask
    f1 = f1_score(masks[real_idx].flatten(), pred_bin.flatten(), zero_division=0)
    all_f1.append(f1)

    # side-by-side comparison
    axes[i][0].imshow(images[real_idx])
    axes[i][0].set_title("Original Pre-Image", fontsize=10, fontweight='bold')
    axes[i][0].axis('off')

    axes[i][1].imshow(masks[real_idx], cmap='gray')
    axes[i][1].set_title("Ground Truth Buildings (OSM)", fontsize=10, fontweight='bold')
    axes[i][1].axis('off')

    # semi-transparent yellow overlay for the predicted buildings
    overlay = images[real_idx].copy()
    yellow  = np.array([1.0, 0.84, 0.0], dtype=np.float32)
    for c in range(3):
        overlay[:,:,c][pred_bin > 0.5] = (
            overlay[:,:,c][pred_bin > 0.5] * 0.4 + yellow[c] * 0.6
        )
    axes[i][2].imshow(overlay)
    axes[i][2].set_title(f"Predicted Buildings \U0001F7E1\nF1-Score: {f1*100:.1f}%", fontsize=10, color='darkgreen', fontweight='bold')
    axes[i][2].axis('off')

mean_f1_total = np.mean(all_f1) * 100
print(f"\nMean F1-Score: {mean_f1_total:.1f}%")
plt.suptitle(f"Building Detection Validation Results — Mean F1-Score: {mean_f1_total:.1f}%", fontsize=13, y=0.98, fontweight='bold')
plt.tight_layout()
plt.savefig("building_results.png", dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# ═══════════════════════════════════════════════
# CELL 4 -- Inference + visualization for the building model
# ═══════════════════════════════════════════════

import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2

SAVE_PATH = '/content/drive/MyDrive/Shati_Project'
device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 1. Load the model
model = smp.Unet(
    encoder_name='resnet34',
    encoder_weights=None,
    in_channels=3,
    classes=1
)
model.load_state_dict(torch.load(f'{SAVE_PATH}/best_building_model.pth', map_location=device))
model = model.to(device)
model.eval()

# 2. Standard inference transform
transform = A.Compose([
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

def predict_and_draw(img_array):
    """Predict + fix mask polarity + draw hollow outlines around buildings."""
    img_u8 = (img_array * 255).astype(np.uint8).copy()
    tensor = transform(image=img_u8)['image'].unsqueeze(0).to(device)

    with torch.no_grad():
        pred = torch.sigmoid(model(tensor)).cpu().squeeze().numpy()

    # convert to a binary mask
    pred_bin = (pred > 0.5).astype(np.uint8)

    # the raw model output marks streets as 1 / rooftops as 0 -- invert it
    pred_bin_corrected = 1 - pred_bin

    # draw hollow yellow outlines precisely around the rooftops
    result = img_u8.copy()
    contours, _ = cv2.findContours(pred_bin_corrected, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    cv2.drawContours(result, contours, -1, (255, 215, 0), thickness=1)

    return result, pred_bin_corrected, len(contours)

# ─── Show 4 examples from the Al-Shati validation split ─────
split = int(len(images) * 0.8)
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

for i in range(2):
    # spaced-out samples, to explore geographic diversity
    idx = split + i * 15
    result, corrected_mask, num_buildings = predict_and_draw(images[idx])

    # column 1: the raw satellite image
    axes[i][0].imshow(images[idx])
    axes[i][0].set_title("Original Pre-Image", fontsize=11, fontweight='bold')
    axes[i][0].axis('off')

    # column 2: the OSM-derived ground-truth mask
    axes[i][1].imshow(masks[idx], cmap='gray')
    axes[i][1].set_title("Ground Truth (OSM)", fontsize=11, fontweight='bold')
    axes[i][1].axis('off')

    # column 3: final result with yellow outlines around detected buildings
    axes[i][2].imshow(result)
    axes[i][2].set_title(f"AI Detected Building Contours\n({num_buildings} buildings identified)", fontsize=11, fontweight='bold', color='green')
    axes[i][2].axis('off')

plt.suptitle("Gaza Damage Assessment System — Stage 1: Building Footprints Extraction", fontsize=14, fontweight='bold', y=0.98)
plt.tight_layout()
plt.savefig("building_contours_corrected.png", dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# ═══════════════════════════════════════════════
# CELL 5 -- Experiment: separating attached buildings (watershed)
# ═══════════════════════════════════════════════
# Because we need to identify buildings individually and Al-Shati's buildings
# are tightly packed together (and our imagery isn't the sharpest), this cell
# tries a watershed-based instance-separation step on top of the binary
# building mask. This is the exact experiment the paper's Appendix A describes
# as contributing to "watershed over/under-segmentation on attached buildings"
# -- kept here as a documented negative result, not a recommended approach.

import cv2
import numpy as np
from scipy import ndimage
import torch
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2

SAVE_PATH = '/content/drive/MyDrive/Shati_Project'
device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 1. Load the building detection model
bldg_model = smp.Unet(
    encoder_name='resnet34', encoder_weights=None,
    in_channels=3, classes=1
).to(device)
bldg_model.load_state_dict(torch.load(f'{SAVE_PATH}/best_building_model.pth', map_location=device))
bldg_model.eval()

def separate_buildings(pred_mask):
    """Separate attached/touching buildings into individual instances."""
    mask = pred_mask.astype(np.uint8)

    # 1. remove noise
    kernel = np.ones((3,3), np.uint8)
    mask   = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)

    # 2. distance transform -- locates each building's center
    dist   = cv2.distanceTransform(mask, cv2.DIST_L2, 5)
    dist   = cv2.normalize(dist, None, 0, 1.0, cv2.NORM_MINMAX)

    # 3. determine sure foreground (building centers)
    _, sure_fg = cv2.threshold(dist, 0.4, 1, cv2.THRESH_BINARY)
    sure_fg    = sure_fg.astype(np.uint8)

    # 4. watershed
    sure_bg  = cv2.dilate(mask, kernel, iterations=3)
    unknown  = cv2.subtract(sure_bg, sure_fg)
    _, markers = cv2.connectedComponents(sure_fg)
    markers  = markers + 1
    markers[unknown == 1] = 0

    img_3ch  = cv2.cvtColor(mask * 255, cv2.COLOR_GRAY2BGR)
    markers  = cv2.watershed(img_3ch, markers)

    return markers

# ─── Run it on one example ────────────────────────
transform = A.Compose([
    A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
    ToTensorV2(),
])

# pick a patch with clearly visible buildings
split = int(len(images) * 0.8)
idx   = split + 20

img_u8 = (images[idx] * 255).astype(np.uint8)
tensor = transform(image=img_u8)['image'].unsqueeze(0).to(device)
with torch.no_grad():
    pred = torch.sigmoid(bldg_model(tensor)).cpu().squeeze().numpy()
pred_bin = (pred > 0.5).astype(np.uint8)

# separate the buildings
markers = separate_buildings(pred_bin)

# draw each separated building in a different color
result = img_u8.copy()
unique = np.unique(markers)
unique = unique[unique > 1]  # ignore background

colors = [(255,215,0),(0,255,0),(0,255,255),(255,128,0),(255,0,255)]
for i, label in enumerate(unique):
    color    = colors[i % len(colors)]
    contours,_ = cv2.findContours(
        (markers == label).astype(np.uint8),
        cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )
    cv2.drawContours(result, contours, -1, color, 2)

print(f"Number of separated buildings: {len(unique)}")

# display
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(images[idx])
axes[0].set_title("Original"); axes[0].axis('off')

axes[1].imshow(pred_bin, cmap='gray')
axes[1].set_title("Building Mask"); axes[1].axis('off')

axes[2].imshow(result)
axes[2].set_title(f"Individual Buildings\n{len(unique)} buildings detected")
axes[2].axis('off')

plt.suptitle("Building Separation — Watershed Algorithm", fontsize=13)
plt.tight_layout()
plt.savefig("buildings_separated.png", dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ═══════════════════════════════════════════════
# CELL 6 -- Prepare damage-detection training data
# ═══════════════════════════════════════════════
# Damage labels here are a heuristic pixel-difference pseudo-label, NOT a
# weak label derived from UNOSAT -- see the final pipeline notebook for that.

import cv2
import numpy as np
import pickle
import os
import matplotlib.pyplot as plt

SAVE_PATH = '/content/drive/MyDrive/Shati_Project'
TARGET_SIZE = 256
STRIDE      = 64
MAX_PATCHES = 3000

# load the before/after pairs
with open(f'{SAVE_PATH}/shati_pairs.pkl', 'rb') as f:
    pairs = pickle.load(f)
print(f"{len(pairs)} pairs")
for p in pairs:
    print(f"   {p['section']} | pre: {p['pre']['timestamp']} | post: {p['post']['timestamp']}")

# helper function: build a pseudo-label damage mask from a before/after pair
def make_damage_mask(pre, post, threshold=0.10):
    pre_f  = pre.astype(np.float32)  / 255.0
    post_f = post.astype(np.float32) / 255.0
    diff   = np.abs(pre_f.mean(axis=2) - post_f.mean(axis=2))
    diff   = cv2.GaussianBlur(diff, (5,5), 0)
    mask   = (diff > threshold).astype(np.uint8)
    kernel = np.ones((5,5), np.uint8)
    mask   = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  kernel)
    mask   = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    # mask out sea/water (false "change" due to waves/glare, not damage)
    blue  = pre_f[:,:,2]; green = pre_f[:,:,1]; red = pre_f[:,:,0]
    water = ((blue > 0.35) | (green > 0.35)) & (blue > red)
    mask[water] = 0
    return mask.astype(np.float32)

# ─── Build the training patches ───────────────────
all_pre   = []
all_post  = []
all_masks = []

print("Building patches...")

for pair in pairs:
    pre_img  = cv2.cvtColor(cv2.imread(pair['pre']['img_path']),  cv2.COLOR_BGR2RGB)
    post_img = cv2.cvtColor(cv2.imread(pair['post']['img_path']), cv2.COLOR_BGR2RGB)

    # match sizes if they differ
    if pre_img.shape != post_img.shape:
        post_img = cv2.resize(post_img, (pre_img.shape[1], pre_img.shape[0]))

    # build the pseudo-label mask
    mask = make_damage_mask(pre_img, post_img)
    print(f"   {pair['section']}: overall damage fraction = {mask.mean()*100:.1f}%")

    if mask.mean() < 0.03:
        print("   low damage signal -- lowering the threshold")
        mask = make_damage_mask(pre_img, post_img, threshold=0.07)

    h, w = pre_img.shape[:2]
    for y in range(0, h - TARGET_SIZE + 1, STRIDE):
        for x in range(0, w - TARGET_SIZE + 1, STRIDE):
            pre_p  = pre_img [y:y+TARGET_SIZE, x:x+TARGET_SIZE]
            post_p = post_img[y:y+TARGET_SIZE, x:x+TARGET_SIZE]
            mask_p = mask    [y:y+TARGET_SIZE, x:x+TARGET_SIZE]

            if 0.05 < mask_p.mean() < 0.90:
                all_pre.append(pre_p.astype(np.float32)/255.0)
                all_post.append(post_p.astype(np.float32)/255.0)
                all_masks.append(mask_p)

    if len(all_masks) >= MAX_PATCHES:
        break

print(f"\nTotal patches: {len(all_masks)}")
print(f"Average damage fraction: {np.mean([m.mean() for m in all_masks])*100:.1f}%")

# ─── Save to Drive ─────────────────────────────────
np.save(f'{SAVE_PATH}/damage_pre.npy',   np.array(all_pre,   dtype=np.float32))
np.save(f'{SAVE_PATH}/damage_post.npy',  np.array(all_post,  dtype=np.float32))
np.save(f'{SAVE_PATH}/damage_masks.npy', np.array(all_masks, dtype=np.float32))
print("Saved to Drive!")

# ─── Example visualization ────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for i in range(2):
    idx = i * 20
    axes[i][0].imshow(all_pre[idx])
    axes[i][0].set_title("Pre");  axes[i][0].axis('off')
    axes[i][1].imshow(all_post[idx])
    axes[i][1].set_title("Post"); axes[i][1].axis('off')
    axes[i][2].imshow(all_masks[idx], cmap='RdYlGn_r')
    axes[i][2].set_title(f"Damage Mask\n{all_masks[idx].mean()*100:.1f}%")
    axes[i][2].axis('off')
plt.suptitle("Damage Detection Data", fontsize=13)
plt.tight_layout()
plt.savefig("damage_data.png", dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Optional: refresh the "post" image to a specific later date ──
# The original pairs use the earliest available post-war image; this snippet
# swaps in a mid/late-2024 snapshot instead (useful for looking at longer-term
# change rather than the immediate aftermath), trying a few candidate dates
# per section.
import pickle
import json
import cv2
from pathlib import Path

SAVE_PATH = '/content/drive/MyDrive/Shati_Project'
DATA_PATH = Path('/content/data/Gaza Before and After')

# load the pairs
with open(f'{SAVE_PATH}/shati_pairs.pkl', 'rb') as f:
    pairs = pickle.load(f)

# look for a specific date's image for a given section
def find_image(section_id, timestamp):
    img_path = DATA_PATH / 'images' / section_id / f'{timestamp}.png'
    if img_path.exists():
        img = cv2.imread(str(img_path))
        if img is not None and img.mean() > 10:
            return {'section_id': section_id, 'timestamp': timestamp,
                    'img_path': str(img_path), 'bbox': pairs[0]['pre']['bbox']}
    return None

# update every pair to use a 2024 post-war image, trying a few candidate dates
for i, pair in enumerate(pairs):
    section = pair['section']
    new_post = find_image(section, '2024-05-26')
    if new_post is None:
        new_post = find_image(section, '2024-07-07')
    if new_post is None:
        new_post = find_image(section, '2024-06-09')

    if new_post:
        pairs[i]['post'] = new_post
        print(f"{section}: post updated to {new_post['timestamp']}")
    else:
        print(f"{section}: no 2024 image found")

# save
with open(f'{SAVE_PATH}/shati_pairs.pkl', 'wb') as f:
    pickle.dump(pairs, f)
print("\nPairs updated and saved!")

# verify
for pair in pairs:
    print(f"   {pair['section']}: pre={pair['pre']['timestamp']} | post={pair['post']['timestamp']}")


In [ ]:
# ═══════════════════════════════════════════════
# CELL 7 -- Train the damage-detection model (6 channels)
# ═══════════════════════════════════════════════

import torch
import torch.nn as nn
import numpy as np
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

SAVE_PATH = '/content/drive/MyDrive/Shati_Project'
device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# load the data
pre_patches  = np.load(f'{SAVE_PATH}/damage_pre.npy')
post_patches = np.load(f'{SAVE_PATH}/damage_post.npy')
damage_masks = np.load(f'{SAVE_PATH}/damage_masks.npy')
print(f"{len(post_patches)} patches ready!")

# Dataset: concatenates the post- and pre-image tensors into one 6-channel input
class DamageDataset(Dataset):
    def __init__(self, pre, post, masks, augment=True):
        self.pre   = pre
        self.post  = post
        self.masks = masks
        self.transform = A.Compose([
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.RandomRotate90(p=0.5),
            A.RandomBrightnessContrast(p=0.3),
            A.GaussNoise(p=0.2),
            A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
            ToTensorV2(),
        ], additional_targets={'image2':'image'}) if augment else A.Compose([
            A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
            ToTensorV2(),
        ], additional_targets={'image2':'image'})

    def __len__(self): return len(self.masks)

    def __getitem__(self, i):
        pre  = (self.pre[i]  * 255).astype(np.uint8)
        post = (self.post[i] * 255).astype(np.uint8)
        mask = self.masks[i].astype(np.float32)
        out  = self.transform(image=post, image2=pre, mask=mask)
        combined = torch.cat([out['image'], out['image2']], dim=0)
        return combined, out['mask'].unsqueeze(0)

# train/val split (80/20)
split    = int(len(post_patches) * 0.8)
train_ds = DamageDataset(pre_patches[:split],  post_patches[:split],  damage_masks[:split],  augment=True)
val_ds   = DamageDataset(pre_patches[split:],  post_patches[split:],  damage_masks[split:],  augment=False)
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=8, shuffle=False, num_workers=2)
print(f"Train: {len(train_ds)} | Val: {len(val_ds)}")

# model: EfficientNet-B4-encoder U-Net, 6 input channels (segmentation_models_pytorch
# builds the 6-channel first conv layer natively -- no manual weight-copy surgery needed)
damage_model = smp.Unet(
    encoder_name    = 'efficientnet-b4',
    encoder_weights = None,
    in_channels     = 6,
    classes         = 1,
).to(device)

criterion = smp.losses.DiceLoss('binary')
bce       = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(damage_model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)

# training loop
best_loss  = float('inf')
train_hist = []
val_hist   = []
print("Starting training...")

for epoch in range(1, 51):
    damage_model.train()
    tr_loss = 0
    for imgs, msks in train_loader:
        imgs, msks = imgs.to(device), msks.to(device)
        optimizer.zero_grad()
        preds = damage_model(imgs)
        loss  = criterion(preds, msks) + 0.5 * bce(preds, msks)
        loss.backward()
        optimizer.step()
        tr_loss += loss.item()
    tr_loss /= len(train_loader)

    damage_model.eval()
    vl_loss = 0
    with torch.no_grad():
        for imgs, msks in val_loader:
            imgs, msks = imgs.to(device), msks.to(device)
            vl_loss += criterion(damage_model(imgs), msks).item()
    vl_loss /= len(val_loader)

    scheduler.step()
    train_hist.append(tr_loss)
    val_hist.append(vl_loss)

    saved = ''
    if vl_loss < best_loss:
        best_loss = vl_loss
        torch.save(damage_model.state_dict(), f'{SAVE_PATH}/best_damage_model.pth')
        saved = '(saved)'

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:02d}/50 | Train: {tr_loss:.4f} | Val: {vl_loss:.4f} {saved}")

print(f"\nDone. Best Val Loss: {best_loss:.4f}")

plt.figure(figsize=(10,4))
plt.plot(train_hist, label='Train Loss', color='#8b5cf6')
plt.plot(val_hist,   label='Val Loss',   color='#f87171')
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.title('Training Curve — Damage Detection')
plt.legend(); plt.grid(alpha=0.3)
plt.savefig('damage_curve.png', dpi=150)
plt.show()


In [ ]:
# ═══════════════════════════════════════════════
# CELL 8 -- Full-image overlay: buildings + damage combined
# ═══════════════════════════════════════════════

import torch
import numpy as np
import cv2
import matplotlib.pyplot as plt
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2
import pickle

SAVE_PATH = '/content/drive/MyDrive/Shati_Project'
device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# load both trained models
bldg_model = smp.Unet(encoder_name='resnet34',        encoder_weights=None, in_channels=3, classes=1).to(device)
dmg_model  = smp.Unet(encoder_name='efficientnet-b4', encoder_weights=None, in_channels=6, classes=1).to(device)
bldg_model.load_state_dict(torch.load(f'{SAVE_PATH}/best_building_model.pth', map_location=device))
dmg_model.load_state_dict(torch.load(f'{SAVE_PATH}/best_damage_model.pth',   map_location=device))
bldg_model.eval(); dmg_model.eval()
print("Both models loaded!")

# inference transforms
t3 = A.Compose([A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)), ToTensorV2()])
t6 = A.Compose([A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)), ToTensorV2()],
               additional_targets={'image2':'image'})

# run a model over a full image via sliding-window patches
def predict_full_image(img, model, channels=3, patch_size=256, stride=128):
    """Run the model over a full image by tiling it into overlapping patches."""
    h, w    = img.shape[:2]
    out_map = np.zeros((h, w), dtype=np.float32)
    count   = np.zeros((h, w), dtype=np.float32)

    for y in range(0, h - patch_size + 1, stride):
        for x in range(0, w - patch_size + 1, stride):
            patch = img[y:y+patch_size, x:x+patch_size]
            patch_u8 = (patch * 255).astype(np.uint8)
            out  = t3(image=patch_u8)
            inp  = out['image'].unsqueeze(0).to(device)
            with torch.no_grad():
                pred = torch.sigmoid(model(inp)).cpu().squeeze().numpy()
            out_map[y:y+patch_size, x:x+patch_size] += pred
            count  [y:y+patch_size, x:x+patch_size] += 1

    count[count == 0] = 1
    return out_map / count

def predict_damage_full(pre, post, patch_size=256, stride=128):
    """Run the damage model over two full before/after images."""
    h, w    = pre.shape[:2]
    out_map = np.zeros((h, w), dtype=np.float32)
    count   = np.zeros((h, w), dtype=np.float32)

    for y in range(0, h - patch_size + 1, stride):
        for x in range(0, w - patch_size + 1, stride):
            pre_p  = (pre [y:y+patch_size, x:x+patch_size] * 255).astype(np.uint8)
            post_p = (post[y:y+patch_size, x:x+patch_size] * 255).astype(np.uint8)
            out    = t6(image=post_p, image2=pre_p)
            inp    = torch.cat([out['image'], out['image2']], dim=0).unsqueeze(0).to(device)
            with torch.no_grad():
                pred = torch.sigmoid(dmg_model(inp)).cpu().squeeze().numpy()
            out_map[y:y+patch_size, x:x+patch_size] += pred
            count  [y:y+patch_size, x:x+patch_size] += 1

    count[count == 0] = 1
    return out_map / count

# load the full-resolution images
with open(f'{SAVE_PATH}/shati_pairs.pkl', 'rb') as f:
    pairs = pickle.load(f)

# run on the second pair (index 1)
pair = pairs[1]
print(f"Processing {pair['section']}...")
print(f"   Pre:  {pair['pre']['timestamp']}")
print(f"   Post: {pair['post']['timestamp']}")

pre_full  = cv2.cvtColor(cv2.imread(pair['pre']['img_path']),  cv2.COLOR_BGR2RGB).astype(np.float32)/255.0
post_full = cv2.cvtColor(cv2.imread(pair['post']['img_path']), cv2.COLOR_BGR2RGB).astype(np.float32)/255.0

# match sizes
if pre_full.shape != post_full.shape:
    post_full = cv2.resize(post_full, (pre_full.shape[1], pre_full.shape[0]))
    post_full = post_full.astype(np.float32)

print(f"   Image size: {pre_full.shape[1]}×{pre_full.shape[0]}")

# run both models
print("Running building detection...")
bldg_map = predict_full_image(pre_full, bldg_model, channels=3)
bldg_bin = (bldg_map > 0.3).astype(np.uint8)
print(f"   Building pixels detected: {bldg_bin.sum()}")

print("Running damage detection...")
dmg_map = predict_damage_full(pre_full, post_full)
dmg_bin = (dmg_map > 0.5).astype(np.uint8)
print(f"   Damaged pixels detected: {dmg_bin.sum()}")

# compute the results
dmg_in_bldg = cv2.bitwise_and(dmg_bin, bldg_bin)
total_bldg  = bldg_bin.sum()
dmg_pct     = (dmg_in_bldg.sum() / total_bldg * 100) if total_bldg > 0 else 0

print(f"\nResults:")
print(f"   Damage fraction inside buildings: {dmg_pct:.1f}%")

# draw the final overlay: post-event image + yellow building outlines + red damage
result = (post_full * 255).astype(np.uint8).copy()

# semi-transparent red for damage
overlay = result.copy()
overlay[dmg_bin == 1] = [255, 0, 0]
result = cv2.addWeighted(overlay, 0.35, result, 0.65, 0)

# yellow building outlines
contours, _ = cv2.findContours(bldg_bin, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
cv2.drawContours(result, contours, -1, (255, 215, 0), thickness=1)

print(f"   Buildings detected: {len(contours)}")

# display
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

axes[0][0].imshow(pre_full)
axes[0][0].set_title(f"(a) Pre-event\n{pair['pre']['timestamp']}", fontsize=11)
axes[0][0].axis('off')

axes[0][1].imshow(post_full)
axes[0][1].set_title(f"(b) Post-event\n{pair['post']['timestamp']}", fontsize=11)
axes[0][1].axis('off')

axes[0][2].imshow(bldg_bin, cmap='gray')
axes[0][2].set_title(f"(c) Buildings Detected\n{len(contours)} buildings", fontsize=11)
axes[0][2].axis('off')

axes[1][0].imshow(dmg_bin, cmap='Reds')
axes[1][0].set_title(f"(d) Damage Areas\n{dmg_bin.mean()*100:.1f}% of image", fontsize=11)
axes[1][0].axis('off')

axes[1][1].imshow(dmg_in_bldg, cmap='Reds')
axes[1][1].set_title(f"(e) Damage IN Buildings\n{dmg_pct:.1f}% of buildings", fontsize=11)
axes[1][1].axis('off')

axes[1][2].imshow(result)
axes[1][2].set_title(
    f"(f) Final Overlay\nYellow=Buildings | Red=Damage\n{dmg_pct:.1f}% damaged",
    fontsize=11, color='darkred', fontweight='bold'
)
axes[1][2].axis('off')

plt.suptitle(
    f"Gaza Damage Assessment — Al-Shati Camp ({pair['section']})\n"
    f"Building Detection F1: 72.5% | Buildings: {len(contours)}",
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.savefig("overlay_final.png", dpi=200, bbox_inches='tight')
plt.show()


## Known limitations of this prototype (why it was superseded)

- Error propagation: damage inference depends on a separately-trained building
  model, so building-detection mistakes silently corrupt the final overlay.
- The damage "label" is a naive pixel-difference heuristic, not derived from
  any damage reference (UNOSAT or otherwise) -- it can't distinguish real
  damage from registration error, shadows, or seasonal change.
- No spatially isolated train/val/test split -- patches are split 80/20
  randomly, which the paper's Sec. 2.3 (citing Gerard et al.) specifically
  identifies as producing an optimistic bias.
- No threshold calibration, no ignore class, no building-level aggregation,
  no GIS/web delivery.

See `notebooks/02_geodamage_ai_pipeline.ipynb` for how each of these was
addressed in the paper's final design.
